In [ ]:
# Copyright 2026 Google LLC
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#     https://www.apache.org/licenses/LICENSE-2.0
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Module 12 — Usage Types (Provisioned Throughput & Shared Model Lineage Quota)

The two ways **capacity and quota** are managed for Claude on the Agent Platform.

Unlike Modules 00–11, these are **account/billing-level** capabilities — Provisioned Throughput must be **purchased**, and quota is **account-level**. They are mostly **configured, not called**, so this module **explains** them and links the docs rather than running inference. **It makes no model calls.** This is the **final module** of the series and builds on 00–11.

## Part B — Provisioned Throughput (PT)

**Provisioned Throughput (PT)** reserves **dedicated capacity** for a **fixed fee** — built for **predictable, high-volume production** workloads. It gives **consistent throughput** and **prioritized processing**, rather than competing for on-demand capacity.

PT is **purchased through Google Cloud** (console or contact sales) — **not** a code call. Once you hold a PT order for a **model + region**, eligible requests **consume** it. Critically, you use the **same `AnthropicVertex` code path** as the rest of the series: **PT changes capacity and billing, not your code.**

For exactly how to direct requests at PT vs on-demand and control **spillover**, see the **"Use Provisioned Throughput"** doc.

**Docs:**
- Overview: https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/provisioned-throughput/overview
- Use PT: https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/provisioned-throughput/use-provisioned-throughput
- Purchase PT: https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/provisioned-throughput/purchase-provisioned-throughput

### PT vs Pay-as-you-go

- **Provisioned Throughput** — **reserved**, **fixed-fee** capacity for steady production traffic.
- **Pay-as-you-go (PayGo)** — **on-demand** capacity, offered in tiers (**Standard / Priority / Flex**) that trade off latency, priority, and cost.

Pointer: https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/deploy/consumption-options

## Part C — Shared Model Lineage Quota

At a high level, **Shared Model Lineage Quota** is quota that is **shared across a model's version lineage**. Instead of managing **separate quota per version**, your quota **carries across** as new versions ship — so a newer Claude version draws on the same shared pool rather than requiring a fresh allocation. This **simplifies quota management** for teams that track model updates.

> ℹ️ Confirm the **precise definition and behavior** (which models/versions share a lineage, and how limits are counted) in the docs, as the exact mechanism can change.

Pointer: https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/quotas

## Part D — Check your quotas

To see current **quotas and usage** for your project:

- **Cloud Console → IAM & Admin → Quotas**, filtered to the **Vertex AI / Agent Platform** service, or
- the **`gcloud`** CLI.

The cell below just **prints** the relevant Console URL and an example `gcloud` command for your project — it doesn't execute anything fragile.

In [ ]:
# Non-fragile: auto-detect the project and PRINT guidance (no inference, no fragile calls).
import subprocess

def _default_project() -> str:
    try:
        out = subprocess.run(
            ["gcloud", "config", "get-value", "project"],
            capture_output=True, text=True, timeout=15,
        ).stdout.strip()
        if out and out != "(unset)":
            return out
    except Exception:
        pass
    return "<YOUR_PROJECT_ID>"

PROJECT_ID = _default_project()
assert PROJECT_ID and PROJECT_ID != "<YOUR_PROJECT_ID>", (
    "PROJECT_ID is still the placeholder. Run `gcloud config set project <id>` "
    "so it auto-detects, or set PROJECT_ID directly."
)

print("Check quotas & usage for your project:\n")
print("1) Cloud Console — Quotas (filter to the Vertex AI API / Agent Platform service):")
print(f"   https://console.cloud.google.com/iam-admin/quotas?project={PROJECT_ID}\n")
print("2) gcloud — list Vertex AI quota metrics for the project:")
print(f"   gcloud alpha services quota list \\")
print(f"     --service=aiplatform.googleapis.com \\")
print(f"     --consumer=projects/{PROJECT_ID}\n")
print("(Exact metric names / commands can change — confirm in the quotas doc linked above.)")

## Part E — Notes & recap

### Notes

- **Provisioned Throughput** — **reserved** capacity for a **fixed fee**, purchased via **console / sales**; for predictable high-volume production.
- **Pay-as-you-go tiers** (Standard / Priority / Flex) — **on-demand** capacity when you don't need a reservation.
- **Shared Model Lineage Quota** — quota shared **across a model's versions**, so it carries forward as new versions ship.
- All of these are **account/region-level** and leave your **inference code unchanged** — you keep using the same `AnthropicVertex` path.
- Where a precise mechanism matters, **confirm specifics in the linked docs**, as they can change.

### Recap

- There are two account-level levers: **how you reserve capacity** (Provisioned Throughput vs Pay-as-you-go tiers) and **how quota is counted** (Shared Model Lineage Quota across versions).
- Both are **configured/purchased**, not invoked in code — your inference path stays the same.
- You can inspect current limits via the **Console Quotas page** or **`gcloud`**.

This completes the Claude-on-Agent-Platform feature series (modules 00–12). See the folder README for the full index.